# Traditional RAG : 
- Why RAG ? 
- comparison: without or with RAG

**Goal:** Ask the **same question** twice. First, a language model answers without access to a fictional company's internal policy. Then, Retrieval-Augmented Generation (RAG) retrieves the policy from a **FAISS vector store** and gives the model the relevant evidence.

**Requirements:** Python, an OpenAI API key, internet access for OpenAI API calls.

> The policy below is fictional and created specifically for this demo. The point is to demonstrate access to information

## 1. Install dependencies
Run once, then restart the kernel if Jupyter asks you to.

In [1]:
# pip install -q
%uv add dotenv langchain-openai langchain-core faiss-cpu

/Users/marcelogomesmarques/Projects/LLM/master_rag/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%uv add langchain-ollama

## 2. Set up the model

In [6]:
import os
import getpass
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI #, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

#llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), temperature=0)
llm = ChatOpenAI(
            model="qwen2.5:7b", #"tencent/hy3:free",
            base_url="http://localhost:11434/v1", #"https://openrouter.ai/api/v1",
            api_key="olama", # os.getenv("OPENROUTER_API_KEY"),
            temperature=0
            )

## 3. Documents Indexing 
These three snippets are **our fictional source documents**. One contains the answer; the others are distractors. No external files or services are needed for storage.

In [8]:
documents = [
    Document(
        page_content="AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.",
        metadata={"source": "IT-LAPTOP-01"},
    ),
    Document(
        page_content="AcmeOrbit travel policy: Expense claims must be submitted within 30 days of travel.",
        metadata={"source": "TRAVEL-02"},
    ),
    Document(
        page_content="AcmeOrbit security policy: Employees must use multifactor authentication for company accounts.",
        metadata={"source": "SECURITY-03"},
    ),
]

# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest", base_url="http://localhost:11434")
vectorstore = InMemoryVectorStore.from_documents(documents, embeddings)

# print(f"FAISS vector store created with {vectorstore.index.ntotal} documents")
print(f"Vector store size: {len(vectorstore.store)} documents")


Vector store criado com 3 documentos


## 4. NO RAG pipeline

The model only sees the question. It should admit it cannot verify the fictional policy, though model behavior can vary.

In [9]:
question = "How many days does AcmeOrbit give employees to return a borrowed laptop?"
print(question)

no_rag = llm.invoke([
    ("system", "Answer accurately. If you do not have the company's policy, say you do not know; do not guess."),
    ("human", question),
])
print("WITHOUT RAG:\n", no_rag.content)

How many days does AcmeOrbit give employees to return a borrowed laptop?
WITHOUT RAG:
 I do not know the specific policy of AcmeOrbit regarding the return of borrowed laptops. Without access to their internal policies, I cannot provide an accurate number of days.


## 5. With RAG
RAG gives an LLM access to relevant external knowledge at answer time—without retraining the model.

1. **Retrieve:** FAISS searches for the most relevant snippet using vector similarity.  
2. **Augment:** Put the retrieved snippet in the prompt.  
3. **Generate:** The *same* model answers using that evidence.

In [10]:
def retrieve_context(question, vectorstore, k=1):
    retrieved_docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in retrieved_docs
    )
    return context

In [11]:

context = retrieve_context(question, vectorstore, k=1)
print("RETRIEVED CONTEXT:\n", context)

with_rag = llm.invoke([
    ("system", "Answer using ONLY the supplied context. Cite the source ID. If the context does not contain the answer, say you do not know."),
    ("human", f"Context:\n{context}\n\nQuestion: {question}"),
])
print("\nWITH RAG:\n", with_rag.content)

RETRIEVED CONTEXT:
 Source: IT-LAPTOP-01
AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.

WITH RAG:
 14 calendar days. Source ID: IT-LAPTOP-01


## 6. Compare the two answers (With vs. No RAG)

In [12]:
print("QUESTION:", question)
print("\nEXPECTED FACT FROM THE DOCUMENT: 14 calendar days (IT-LAPTOP-01)")

print("\nWITHOUT RAG:", no_rag.content)
print("\nWITH RAG:", with_rag.content)

QUESTION: How many days does AcmeOrbit give employees to return a borrowed laptop?

EXPECTED FACT FROM THE DOCUMENT: 14 calendar days (IT-LAPTOP-01)

WITHOUT RAG: I do not know the specific policy of AcmeOrbit regarding the return of borrowed laptops. Without access to their internal policies, I cannot provide an accurate number of days.

WITH RAG: 14 calendar days. Source ID: IT-LAPTOP-01


## What did RAG add?

| Without RAG                                      | With RAG                                           |
|--------------------------------------------------|----------------------------------------------------|
| Model receives only the question                 | Model receives the question and retrieved evidence |
| Cannot reliably know private, fictional policies | Can answer from the supplied company policy        |
| May abstain or guess                             | Can cite `IT-LAPTOP-01`                            |


### LLM only
![LLM](llm.png)

### with RAG
![RAG](rag.png)

---

## Exercise 

- Change the question to: **“When must employees submit travel expenses?”** Then rerun the *without RAG*, retrieval, and *with RAG* cells. 
- The expected answer is **within 30 days of travel**, citing `TRAVEL-02`.

In [14]:
ex_question = "When must employees submit travel expenses?"
prompt = "Answer using ONLY the supplied context. Cite the source ID. If the context does not contain the answer or is incomplete or missing, say you do not know."
ex_context = retrieve_context(ex_question, vectorstore, k=1)

no_rag = llm.invoke([
    ("system", 
     "You are answering questions about AcmeOrbit's internal policies. "
    "You have not been given access to those policies. "
    "If you cannot verify the answer, respond exactly: I don't know."),
    ("human", f"Question: {ex_question}"),
])
with_rag = llm.invoke([
    ("system", 
      "Answer using ONLY the provided context. "
      "If the context does not contain the answer, "
      "respond exactly: I don't know. "
      "Otherwise, cite the source ID."),
    ("human", f"Context:\n{ex_context}\n\nQuestion: {ex_question}"),
])


## Result

In [15]:
print("QUESTION:", ex_question)
print("\nThe expected answer is **within 30 days of travel**, citing `TRAVEL-02`")

print("\nWITHOUT RAG:", no_rag.content)
print("\nWITH RAG:", with_rag.content)

QUESTION: When must employees submit travel expenses?

The expected answer is **within 30 days of travel**, citing `TRAVEL-02`

WITHOUT RAG: I don't know.

WITH RAG: Within 30 days of travel. Source ID: TRAVEL-02
